Download Asset Data From Yahoo

In [ ]:
%pip install yfinance
import yfinance as yf
def download_assets(tickers, start_date, end_date):
    assets = yf.download(
        tickers,
        start=start_date,
        end=end_date,
        auto_adjust=False
    )["Adj Close"]

    return assets.dropna(axis=1, how="all").dropna()

def download_benchmark(start_date, end_date):
    benchmark = yf.download(
        "^GSPC",
        start=start_date,
        end=end_date,
        auto_adjust=False
    )["Adj Close"]

    return benchmark.dropna()

Optimal Weight & Age Adjustment

In [ ]:
import pandas as pd
def download_rf(start_date, end_date):
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS3"

    rf = pd.read_csv(
        url,
        index_col=0,
        parse_dates=True,
        na_values="."
    )

    rf = rf.loc[start_date:end_date, "DGS3"].dropna()

    return rf / 100
def risky_allocation(expected_return, rf, variance, A, age):
    base_weight = (expected_return - rf) / (A * variance)

    if age < 40:
        age_factor = 1.00
    elif 40<=age < 60:
        age_factor = 0.85
    else:
        age_factor = 0.70

    return base_weight * age_factor

# We apply an age adjustment to reduce risky-asset exposure for older
# investors, assuming they have less time to recover from potential losses.

Calculating Returns & Variances

In [ ]:
# Daily simple returns for individual assets
def calculate_returns(assets):
    returns = assets.pct_change(fill_method=None).dropna()
    return returns


# Historical annualized mean returns for individual assets
def annualized_returns_individual(assets):
    returns = calculate_returns(assets)
    return returns.mean() * 252

# Annualized variance for individual assets
def annualized_variance_individual(assets):
    returns = calculate_returns(assets)
    return returns.var() * 252

# Annualized covariance matrix
def annualized_covariance(assets):
    returns = calculate_returns(assets)
    return returns.cov() * 252


# Daily simple returns for the benchmark
def benchmark_returns(benchmark):
    return calculate_returns(benchmark)


# Historical annualized mean return for the benchmark
def annualized_benchmark_returns(benchmark):
    returns = benchmark_returns(benchmark)
    return returns.mean() * 252


# Annualized variance of the benchmark
def annualized_benchmark_variance(benchmark):
    returns = benchmark_returns(benchmark)
    return returns.var() * 252

CAPM Estimation

In [ ]:
import statsmodels.api as sm

def calculate_betas(assets, benchmark, rf):
    asset_returns = calculate_returns(assets)
    market_returns = calculate_returns(benchmark).squeeze()
    
    rf_daily = rf / 252

    betas = {}

    for ticker in asset_returns.columns:
        CAPM = pd.concat([asset_returns[ticker].rename("asset_return"), market_returns.rename("market_return"), rf_daily.rename("rf")], axis=1).dropna()

        CAPM["market_excess"] = (CAPM["market_return"] - CAPM["rf"])

        CAPM["asset_excess"] = ( CAPM["asset_return"] - CAPM["rf"])

        x_CAPM = sm.add_constant(CAPM["market_excess"])

        model = sm.OLS(CAPM["asset_excess"], x_CAPM).fit()

        betas[ticker] = model.params["market_excess"]

    return pd.Series(betas, name="beta")

def expected_returns(assets, benchmark, rf):
    betas = calculate_betas(assets, benchmark, rf)

    market_return = annualized_benchmark_returns(benchmark).item()
    rf_annual = rf.iloc[-1]

    capm_returns = rf_annual + betas * (market_return - rf_annual)

    return capm_returns